# Capstone Gate 4 (Part 3): The Note Writer, the Checker and the Fallback
Goal: turn the tier and the top reasons into one short note for the coach, and make sure a bad note never reaches the coach.

**The idea in 4 steps:**
1. We already have the tier and the reasons (from Part 1 and Part 2).
2. The **LLM** writes a short note from them.
3. The **checker** tests the note with 6 simple rules.
4. If the note fails any rule, or the LLM does not work, we use a **fixed template note** instead (the fallback).

So the coach always gets a safe note.

In [1]:
# Import the libraries we need
import pandas as pd
import re
import json

## Part A: Load the Tiers and Reasons

In [2]:
# Load the table we saved in Part 2
df = pd.read_csv("drivers_output.csv")
print(df.shape)
print(df.head(5))

(1674, 6)
   Athlete_ID  Day    Tier Predicted_Tier  Number_Of_Drivers  \
0        1005    1   Green          Green                  1   
1        1005    2  Orange         Orange                  1   
2        1005    3   Green          Green                  0   
3        1005    4     Red            Red                  2   
4        1005    5  Orange         Orange                  2   

                                             Drivers  
0              Sleep is low: 6.9 vs team average 7.5  
1              Sleep is low: 7.3 vs team average 7.5  
2                                                NaN  
3  Sleep is low: 6.4 vs team average 7.5; Stress ...  
4  Sleep is low: 6.8 vs team average 7.5; Muscle ...  


In [3]:
# Turn one row into a "case" (this is exactly what the LLM will see)
bands = {"Green": "75-100", "Orange": "40-74", "Red": "0-39"}

def make_case(row):
    if pd.isna(row["Drivers"]):
        drivers = []
    else:
        drivers = row["Drivers"].split("; ")

    return {
        "athlete": "A" + str(row["Athlete_ID"]),
        "day": int(row["Day"]),
        "tier": row["Predicted_Tier"],
        "band": bands[row["Predicted_Tier"]],
        "drivers": drivers,
    }

# Example
example = make_case(df[df["Predicted_Tier"] == "Red"].iloc[3])
print(example)

{'athlete': 'A1005', 'day': 18, 'tier': 'Red', 'band': '0-39', 'drivers': ['Muscle soreness is high: 8.0 vs team average 5.1', 'Sleep is low: 6.9 vs team average 7.5']}


## Part B: The Fallback Note (no LLM)

In [4]:
# What a coach should do for each tier
action_by_tier = {
    "Green": "Full training load and match readiness.",
    "Orange": "Modified training volume and targeted recovery; monitor closely.",
    "Red": "Active rest and a check with the medical or athletic therapy staff.",
}

# A fixed template note
def fallback_note(case):
    if case["drivers"]:
        reasons = "Main reasons: " + "; ".join(case["drivers"]) + "."
    else:
        reasons = "No single clear reason."

    return {
        "summary": f"{case['athlete']} is in the {case['tier']} tier. {reasons}",
        "action": action_by_tier[case["tier"]],
    }

print(fallback_note(example))

{'summary': 'A1005 is in the Red tier. Main reasons: Muscle soreness is high: 8.0 vs team average 5.1; Sleep is low: 6.9 vs team average 7.5.', 'action': 'Active rest and a check with the medical or athletic therapy staff.'}


## Part C: The Checker (6 Rules)

A note passes only if **all 6 rules** are true:
1. It has both parts (`summary` and `action`)
2. It is under 60 words
3. It names the right tier, and no other tier
4. Every number in the note was in the input (no made-up numbers)
5. It has no medical words (diagnosis, injury, and so on)
6. A Red note tells the coach to check with the medical or athletic therapy staff

In [5]:
# Is this word in the text? (whole word only, so "reduce" does not count as "red")
def has_word(text, word):
    return re.search(r"\b" + word.lower() + r"\b", text.lower()) is not None

banned_words = ["diagnosis", "diagnose", "injury", "injured", "torn", "fracture", "medication", "sprain"]

# Return the list of rules the note breaks (empty list = the note is fine)
def check_note(note, case):
    # Rule 1: both parts
    if not isinstance(note, dict) or "summary" not in note or "action" not in note:
        return ["missing_part"]

    text = note["summary"] + " " + note["action"]
    problems = []

    # Rule 2: length
    if len(text.split()) > 60:
        problems.append("too_long")

    # Rule 3: the right tier only
    if not has_word(text, case["tier"]):
        problems.append("tier_missing")
    for other in ["Green", "Orange", "Red"]:
        if other != case["tier"] and has_word(text, other):
            problems.append("wrong_tier")

    # Rule 4: no made-up numbers
    allowed = re.findall(r"\d+(?:\.\d+)?", json.dumps(case))
    for number in re.findall(r"\d+(?:\.\d+)?", text):
        if number not in allowed:
            problems.append("made_up_number")
            break

    # Rule 5: no medical words
    if any(has_word(text, w) for w in banned_words):
        problems.append("medical_word")

    # Rule 6: Red must mention the medical or therapy staff
    if case["tier"] == "Red" and not re.search(r"medical|therap", text.lower()):
        problems.append("no_referral")

    return problems

print("Fallback note passes?", check_note(fallback_note(example), example) == [])

Fallback note passes? True


**Test the checker:** we write bad notes on purpose and see that it catches them.

In [6]:
# Bad notes made on purpose
bad_1 = {"summary": "A1 is in the Red tier. Sleep was 4.2 hours.", "action": "See the medical staff."}      # made-up number
bad_2 = {"summary": "A1 is in the Green tier.", "action": "Full training load."}                          # wrong tier
bad_3 = {"summary": "A1 is in the Red tier.", "action": "Possible injury, see the medical staff."}        # medical word
bad_4 = {"summary": "A1 is in the Red tier.", "action": "Rest today."}                                     # no referral

test_case = {"athlete": "A1", "day": 1, "tier": "Red", "band": "0-39", "drivers": []}
for bad in [bad_1, bad_2, bad_3, bad_4]:
    print(check_note(bad, test_case))

['made_up_number']
['tier_missing', 'wrong_tier', 'no_referral']
['medical_word']
['no_referral']


## Part D: The LLM Note

**Important:** we use a switch.
- `use_ollama = False`: a **pretend LLM** writes the notes. It is only for testing that our checker and fallback work. It breaks about 1 note in 4 on purpose. **Never report its results as real LLM results.**
- `use_ollama = True`: a real model (Llama 3.2) runs on your own laptop through Ollama.

To turn it on: install Ollama from ollama.com, then in the terminal run `ollama pull llama3.2` and `pip install ollama`, keep Ollama running, and set `use_ollama = True` below.

In [7]:
# Switch: False = pretend LLM (for testing), True = real Ollama model
use_ollama = False

# The instructions we give the LLM
system_prompt = (
    "You write short readiness notes for athletic coaching staff. "
    "Use only the tier and reasons in the input. "
    "Do not add numbers or facts that are not in the input. "
    "Do not give medical diagnoses or mention injuries. "
    "For a Red tier, tell the coach to check with the medical or athletic therapy staff. "
    "Reply in JSON with two fields: summary (1 or 2 sentences) and action (1 sentence). "
    "Keep it under 60 words."
)

In [8]:
# The real LLM (needs Ollama)
def ollama_note(case):
    import ollama
    reply = ollama.chat(
        model="llama3.2",
        format="json",
        options={"temperature": 0},
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user", "content": json.dumps(case)}],
    )
    return json.loads(reply["message"]["content"])

In [9]:
# The pretend LLM (for testing only): a normal note, but it breaks about 1 in 4 on purpose
def pretend_llm(case):
    reasons = "; ".join(case["drivers"])
    note = {
        "summary": f"{case['athlete']} is in the {case['tier']} tier because: {reasons}.",
        "action": action_by_tier[case["tier"]],
    }

    # Pick a "mistake" in a repeatable way (same case = same mistake)
    r = (int(case["athlete"][1:]) + case["day"]) % 16
    if r == 0:
        note["summary"] += " Sleep was 4.2 hours."                  # made-up number
    elif r == 1:
        other = "Green" if case["tier"] != "Green" else "Red"
        note["summary"] = note["summary"].replace(case["tier"], other)  # wrong tier
    elif r == 2:
        note["action"] = "Possible injury, consider a diagnosis."   # medical words
    elif r == 3:
        return {"summary": note["summary"]}                         # missing part
    return note

# Use whichever one the switch says
def llm_note(case):
    if use_ollama:
        return ollama_note(case)
    return pretend_llm(case)

## Part E: One Safe Function

In [10]:
# Try the LLM, check the note, and use the fallback if anything goes wrong
def get_note(case, use_llm=True):
    problems = []

    if use_llm and case["drivers"]:
        try:
            note = llm_note(case)
            problems = check_note(note, case)
            if problems == []:
                return note, "LLM", []
        except Exception:
            problems = ["llm_error"]     # the LLM did not work at all

    # Fallback: the fixed template note
    return fallback_note(case), "fallback", problems

In [11]:
# Try it on 3 cases
for i in [3, 10, 25]:
    case = make_case(df[df["Predicted_Tier"] == "Red"].iloc[i])
    note, source, problems = get_note(case)
    print(source, "| problems:", problems)
    print(note["summary"])
    print(note["action"])
    print()

LLM | problems: []
A1005 is in the Red tier because: Muscle soreness is high: 8.0 vs team average 5.1; Sleep is low: 6.9 vs team average 7.5.
Active rest and a check with the medical or athletic therapy staff.

LLM | problems: []
A1007 is in the Red tier because: HRV is low: 65.0 vs team average 73.9; Sleep was not logged.
Active rest and a check with the medical or athletic therapy staff.

LLM | problems: []
A1009 is in the Red tier because: Sleep is low: 6.9 vs team average 7.5; Muscle soreness is high: 7.4 vs team average 5.1; HRV is low: 69.0 vs team average 73.9.
Active rest and a check with the medical or athletic therapy staff.



## Part F: Build the Test Set

Gate 4 asks for a labelled test set of 30 to 50 cases. We take **12 cases from each tier** (36 in total) from the test athletes.

In [12]:
# 12 cases from each predicted tier
parts = []
for tier in ["Red", "Orange", "Green"]:
    parts.append(df[df["Predicted_Tier"] == tier].sample(12, random_state=3))
cases_df = pd.concat(parts).reset_index(drop=True)

# How many reasons does each case have?
print(pd.crosstab(cases_df["Predicted_Tier"], cases_df["Number_Of_Drivers"]))

Number_Of_Drivers  0  1  2  3
Predicted_Tier               
Green              9  3  0  0
Orange             3  6  3  0
Red                0  1  3  8


## Part G: Run the Test Set

In [13]:
# Run every case and save the result
rows = []
for i, row in cases_df.iterrows():
    case = make_case(row)
    note, source, problems = get_note(case)

    rows.append({
        "Case": i + 1,
        "Athlete": case["athlete"],
        "Day": case["day"],
        "Tier": case["tier"],
        "Reasons": row["Number_Of_Drivers"],
        "Source": source,
        "Problems": ", ".join(problems),
        "Summary": note["summary"],
        "Action": note["action"],
    })

results = pd.DataFrame(rows)

# How many notes came from the LLM and how many from the fallback?
print(results["Source"].value_counts())
print()

# Which rules were broken?
print(results[results["Problems"] != ""]["Problems"].value_counts())

Source
LLM         19
fallback    17
Name: count, dtype: int64

Problems
made_up_number               2
medical_word, no_referral    1
missing_part                 1
tier_missing, wrong_tier     1
Name: count, dtype: int64


In [14]:
# Check every FINAL note (LLM or fallback) with the same 6 rules
final_problems = 0
for i, row in cases_df.iterrows():
    case = make_case(row)
    note = {"summary": results.loc[i, "Summary"], "action": results.loc[i, "Action"]}
    if check_note(note, case) != []:
        final_problems += 1

print("Final notes that break a rule:", final_problems)

Final notes that break a rule: 0


In [15]:
# Run the same cases with the LLM switched OFF
off_sources = []
for i, row in cases_df.iterrows():
    note, source, problems = get_note(make_case(row), use_llm=False)
    off_sources.append(source)

print(pd.Series(off_sources).value_counts())

fallback    36
Name: count, dtype: int64


**Answer:** *What does this test show?*

- **Careful:** this run used the **pretend LLM**, so these numbers show that our checker and fallback work. They do NOT show how good a real LLM is.
- **19 of 36 notes** passed all 6 rules and came from the (pretend) LLM. **17 used the fallback:** 12 of them had no reasons to explain (we skip the LLM then), and 5 were broken notes that the checker caught (2 made-up numbers, 1 wrong tier, 1 medical words with no referral, 1 missing part).
- **No broken note reached the coach.** Every final note, from the LLM or the fallback, passes the 6 rules.
- **With the LLM switched off,** all 36 cases still get a note. The app works without the LLM.
- **To do:** install Ollama, set `use_ollama = True`, run again, and write the real pass rate here.

## Part H: Reviewers and Save

Both of you read the notes and score them. The last 4 columns are empty on purpose, for you to fill in:
- **Score** from 1 to 5 (is the note clear and useful?)
- **Act?** Yes or No (would a coach act on this note?)

In [16]:
# Empty columns for the two reviewers
results["Reviewer1_Score"] = ""
results["Reviewer1_Act"] = ""
results["Reviewer2_Score"] = ""
results["Reviewer2_Act"] = ""

results.to_csv("labelled_cases.csv", index=False)
print("Saved labelled_cases.csv with", len(results), "cases")

Saved labelled_cases.csv with 36 cases


## Summary (for the Gate 4 slide)

| Item | Result |
|---|---|
| Idea | The LLM writes a short note from the tier and reasons. A checker tests it. A fixed template (fallback) replaces any bad or missing note. |
| LLM | Llama 3.2 through Ollama (free, runs on our laptop). A switch, `use_ollama`, turns it on. |
| Checker | 6 rules: both parts, under 60 words, right tier only, no made-up numbers, no medical words, Red tells the coach to see medical or therapy staff |
| Test set | 36 cases (12 per tier). Both of us score the notes from 1 to 5 and say whether a coach would act on them. |
| Test with the pretend LLM | 19 notes passed, 17 used the fallback (12 had no reasons, 5 were caught). All 36 final notes pass the rules. With the LLM off, all 36 use the fallback. |
| Still to do | Run with real Ollama, record the real pass rate, and both reviewers score the notes |
| Limitations | The pretend LLM results are not real LLM results. The Ollama code is not tested yet. 36 cases is a small test set. |